In [ ]:
import pandas as pd
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal, Optional
import json

# --- 1. CONFIGURACIÓN ---
# ¡OJO! Reemplaza esto con tu clave real
API_KEY = "pedirmela (Diego)" 

SYSTEM_PROMPT = """
Eres un Analista de Datos Experto para Amazon. Tu trabajo es extraer especificaciones técnicas estructuradas de títulos de productos para un modelo de Machine Learning (XGBoost).
Reglas:
1. Sé preciso con los números. Si dice '1TB', storage_gb es 1024.0.
2. Identifica el 'market_tier' basándote en la marca y palabras clave (Pro, Ultra, Basics, Lite).
3. Si el producto es un pack (ej. 2-Pack), indica pack_count=2.
4. Clasifica estrictamente en una de las 12 categorías permitidas.
5. SMART HOME: Timbres (doorbells) y cámaras de seguridad van en 'Networking & Smart Home'.
6. WEARABLES: Solo relojes y pulseras que se llevan puestos.
7. RESOLUCIÓN: Extrae 'resolution_standard' solo para pantallas o cámaras.
"""

client = OpenAI(api_key=API_KEY)

# --- 2. DEFINICIÓN DEL ESQUEMA (TU MODELO MAESTRO) ---
class AmazonFinalSpecs(BaseModel):
    # 1. Taxonomía (Sin cambios)
    category: Literal[
        "Smartphones & Wearables", "Tablets & E-Readers", "Laptops & Chromebooks",
        "Desktops, Workstations & Servers", "PC Components (Core)", "Peripherals & Input Devices",
        "Displays & Mounting", "Audio & Video Equipment", "Cameras & Photography",
        "Networking & Smart Home", "Gaming Systems & VR", "Accessories & Consumables"
    ]

    # 2. Anclas de Valor
    market_tier: Literal["Budget", "Mainstream", "Premium", "Enterprise/Professional"]
    condition: Literal["New", "Renewed/Refurbished"]
    is_premium_brand: bool
    tech_generation: Literal["Cutting-Edge", "Current-Gen", "Last-Gen", "Legacy"]

    # 3. Especificaciones Numéricas (Densas)
    ram_gb: Optional[float] = Field(None, description="RAM o VRAM del sistema en GB")
    storage_gb: Optional[float] = Field(None, description="Capacidad almacenamiento en GB")
    size_value: Optional[float] = Field(None, description="Pulgadas (Screens) o mm (Lentes/Audio)")
    
    # --- AQUÍ ESTÁ EL CAMBIO IMPORTANTE ---
    # Performance value sigue sirviendo para Hz, DPI, MP (Megapixeles)
    performance_value: Optional[float] = Field(None, description="Hz (Monitor), DPI (Mouse), MP (Cámara), Read Speed (MB/s)")
    power_wattage: Optional[float] = Field(None, description="Watios (W)")

    # 4. Descriptores Categóricos (NUEVO: RESOLUCIÓN)
    resolution_standard: Optional[Literal["HD/HD+", "FHD (1080p)", "2K/QHD (1440p)", "4K/UHD (2160p)", "5K/8K+"]] = Field(
        None, description="Estándar de resolución visual. Solo para pantallas, cámaras (video), monitores o proyectores."
    )
    
    cpu_gpu_tier: Optional[str] = Field(None, description="Ej: i7, Ryzen 5, RTX 4060, M3")
    connectivity: Optional[str] = Field(None, description="WiFi 6, 5G, Bluetooth, Wired, PoE")
    
    brand: str
    pack_count: int = Field(1, description="Número de unidades en el paquete")
    confidence: float = Field(description="Confianza en la extracción (0-1)")

# --- 3. DATOS DE PRUEBA (Muestra variada de tu archivo) ---
titulos_prueba = [
    # 1. Laptop con CPU y RAM específica (Prueba para Laptops & Chromebooks)
    "Lenovo Ideapad 3 Laptop, 15.6\" HD Touchscreen, 11th Gen Intel Core i3-1115G4, 12GB DDR4 RAM, 512GB PCIe SSD",
    
    # 2. Consola de Videojuegos (Prueba para Gaming Systems & VR)
    "Xbox Series X 1TB Gaming Console + 1 Wireless Controller - True 4K Gaming, Up to 120 FPS",
    
    # 3. Monitor de alta resolución y Hz (Prueba para Displays & Mounting + Resolution_standard)
    "ASUS ROG Strix 32” 4K HDR Gaming Monitor (XG32UCG) – Dual Mode (4K 160Hz/FHD 320Hz), 0.3ms, Fast IPS",
    
    # 4. Periférico Gaming (Prueba para Peripherals & Input Devices + DPI)
    "Logitech MX Master 3S for Business, Wireless Mouse with Quiet Clicks, 8K DPI, Bluetooth, USB-C",
    
    # 5. Equipo de Audio Profesional (Prueba para Audio & Video Equipment)
    "Sennheiser HD 600 - Audiophile Hi-Res Open Back Dynamic Headphone",
    
    # 6. Almacenamiento NAS / Servidor (Prueba para Desktops, Workstations & Servers)
    "Synology 2-Bay DiskStation DS224+ (Diskless)",
    
    # 7. Wearable real (Para contrastar con el timbre inteligente)
    "Xiaomi Smart Band 9 Global Version (2024) 1.62\" Amoled Display, 233 mAh Battery, BT 5.4",
    
    # 8. Impresora (Prueba para Accessories & Consumables o Peripherals según volumen)
    "Canon PIXMA TR8620a - All-in-One Printer Home Office | Copier | Scanner | Fax | Airprint",
    
    # 9. Lente de Cámara Profesional (Prueba para Cameras & Photography + mm)
    "Sony FE 16-35mm F2.8 GM II",
    
    # 10. Software / Juego físico (Prueba de Gaming Systems & VR)
    "Final Fantasy VII and Final Fantasy VIII Remastered - Twin Pack (Nintendo Switch)"
]

# --- 4. EJECUCIÓN ---
print("🧪 Iniciando prueba con 5 productos...\n")
resultados = []

for titulo in titulos_prueba:
    try:
        completion = client.beta.chat.completions.parse(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Analiza este título: {titulo}"}
            ],
            response_format=AmazonFinalSpecs,
        )
        
        datos = completion.choices[0].message.parsed.dict()
        datos['titulo_original'] = titulo[:30] + "..." # Acortar para visualizar
        resultados.append(datos)
        print(f"✅ Procesado: {datos['brand']} - {datos['category']}")
        
    except Exception as e:
        print(f"❌ Error con '{titulo}': {e}")

# --- 5. VISUALIZACIÓN ---
df_test = pd.DataFrame(resultados)

# Reordenar columnas para ver lo importante primero
cols = ['titulo_original', 'category', 'market_tier', 'ram_gb', 'storage_gb', 'power_wattage', 'pack_count', 'confidence']
df_test = df_test[cols + [c for c in df_test.columns if c not in cols]]

# Mostrar el DataFrame
df_test

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.
🧪 Iniciando prueba con 5 productos...



C:\Users\diego\AppData\Local\Temp\ipykernel_3624\3817738026.py:111: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  datos = completion.choices[0].message.parsed.dict()


✅ Procesado: Lenovo - Laptops & Chromebooks
✅ Procesado: Xbox - Gaming Systems & VR
✅ Procesado: ASUS - Displays & Mounting
✅ Procesado: Logitech - Peripherals & Input Devices
✅ Procesado: Sennheiser - Audio & Video Equipment
✅ Procesado: Synology - Desktops, Workstations & Servers
✅ Procesado: Xiaomi - Smartphones & Wearables
✅ Procesado: Canon - Accessories & Consumables
✅ Procesado: Sony - Cameras & Photography
✅ Procesado: Square Enix - Gaming Systems & VR


,titulo_original,category,market_tier,ram_gb,storage_gb,power_wattage,pack_count,confidence,condition,is_premium_brand,tech_generation,size_value,performance_value,resolution_standard,cpu_gpu_tier,connectivity,brand
0,"Lenovo Ideapad 3 Laptop, 15.6""...",Laptops & Chromebooks,Mainstream,12.0,512.0,None,1,0.95,New,False,Current-Gen,15.60,NaN,HD/HD+,i3-1115G4,None,Lenovo
1,Xbox Series X 1TB Gaming Conso...,Gaming Systems & VR,Premium,NaN,1024.0,None,1,1.00,New,True,Current-Gen,NaN,NaN,4K/UHD (2160p),Unknown,Wired,Xbox
2,ASUS ROG Strix 32” 4K HDR Gami...,Displays & Mounting,Premium,NaN,NaN,None,1,1.00,New,True,Current-Gen,32.00,160.0,4K/UHD (2160p),None,None,ASUS
3,Logitech MX Master 3S for Busi...,Peripherals & Input Devices,Mainstream,NaN,NaN,None,1,1.00,New,True,Current-Gen,NaN,8000.0,None,None,"Bluetooth, USB-C",Logitech
4,Sennheiser HD 600 - Audiophile...,Audio & Video Equipment,Premium,NaN,NaN,None,1,0.95,New,True,Current-Gen,NaN,NaN,None,None,None,Sennheiser
5,Synology 2-Bay DiskStation DS2...,"Desktops, Workstations & Servers",Mainstream,NaN,NaN,None,2,0.90,New,True,Current-Gen,NaN,NaN,None,None,None,Synology
6,Xiaomi Smart Band 9 Global Ver...,Smartphones & Wearables,Mainstream,NaN,NaN,None,1,0.95,New,False,Current-Gen,1.62,NaN,None,None,Bluetooth 5.4,Xiaomi
7,Canon PIXMA TR8620a - All-in-O...,Accessories & Consumables,Mainstream,NaN,NaN,None,1,0.90,New,False,Current-Gen,NaN,NaN,None,None,Airprint,Canon
8,Sony FE 16-35mm F2.8 GM II...,Cameras & Photography,Premium,NaN,NaN,None,1,0.90,New,True,Current-Gen,NaN,NaN,None,None,None,Sony
9,Final Fantasy VII and Final Fa...,Gaming Systems & VR,Mainstream,NaN,NaN,None,2,0.95,New,False,Current-Gen,NaN,NaN,None,None,None,Square Enix


In [71]:
import pandas as pd


ev3 = pd.read_csv('amazon_specs_enriched.csv')
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
ev3['category'].value_counts()

category
Accessories & Consumables           2791
Audio & Video Equipment             1200
Peripherals & Input Devices         1063
Networking & Smart Home              491
Displays & Mounting                  462
Cameras & Photography                457
PC Components (Core)                 455
Laptops & Chromebooks                334
Desktops, Workstations & Servers     188
Smartphones & Wearables              154
Tablets & E-Readers                   78
Gaming Systems & VR                   41
ERROR_API_FAIL                         3
Name: count, dtype: int64

In [23]:
import pandas as pd
import numpy as np
from IPython.display import Image, HTML

# 1. Carga tus datos (ajusta los nombres de tus archivos)
df_specs = pd.read_csv('amazon_specs_enriched.csv')
df_prices = pd.read_csv('../../Datasets/evaluacion2.csv')

# 2. Une ambos datasets (usando el ID o el Título como llave)
df_final = pd.merge(
    df_specs, 
    df_prices, 
    left_on='original_title', 
    right_on='product_title'
)

# 3. Revertir el log1 para ver el precio real
# Si usaste log natural (ln(1+x)):
df_final['price_real'] = np.exp(df_final['log_original_price']) - 1

# Si usaste log base 10 (log10(1+x)):
# df['price_real'] = (10 ** df['log1_price']) - 1

# 4. Filtrar por la categoría Accesorios y ordenar por precio de mayor a menor
expensive_accessories = df_final[df_final['category'] == 'Accessories & Consumables'].sort_values(by='price_real', ascending=False)

# 5. Mostrar el Top 20
print("AUDITORÍA DE ACCESORIOS CAROS:")




# 1. Definimos una función que envuelve la URL en una etiqueta HTML <img>
def render_image(url):
    return f'<img src="{url}" width="80" >'

# 2. Seleccionamos la muestra
sample_peripherals = expensive_accessories[['original_title', 'price_real']].head(10)

# 3. Mostramos la tabla renderizando el HTML
# 'escape=False' permite que el navegador lea los tags <img> en lugar de verlos como texto
HTML(sample_peripherals.to_html(escape=False, formatters=dict(product_image_url=render_image)))

AUDITORÍA DE ACCESORIOS CAROS:


,original_title,price_real
7279,Motorola Solutions RMU2040 6-Pack Two-Way Radio Digital Non-Display 99 UHF Business Exclusive Frequencies,1512.00
3012,Texas Instruments TI- 84Plus CE Teacher's 10 Pack Graphing Calculator,1500.61
4299,"Fellowes Powershred 225Ci 22-Sheet 100% Jam-Proof Crosscut Paper Shredder Commercial Grade for Office, Black 3825001",1045.25
5956,"Garmin Catalyst, Driving Performance Optimizer with Real-time Coaching and Immediate Track Session Analysis, for Motorsports and High Performance Driving (010-02345-00) , Black , 6.95 inch",999.99
1184,"Escort Redline 360c Plug and Play Radar Detector - Extreme Range, Rapid Response Times, Full Stealth, 360 Degree Awareness, Advanced Filtering, Built-in WiFi, Apple CarPlay & Android Auto Compatible",799.95
1538,"HP 508A Cyan, Magenta, Yellow Toner Cartridges (3-pack) | Works with HP Color LaserJet Enterprise M552, M553, HP Color LaserJet Enterprise MFP M577 Series | CF360AM",771.89
3489,"Hp Printing 972X Genuine PageWide Color and Black High Yield Toner Set (F6T84AN, L0R98AN, L0S01AN, L0S04AN)",764.86
463,"Escort MAX 360c MKII Radar and Laser Detector & Escort M2 Smart Dash Cam Bundle - 1080P Full HD Video, Extreme Range, False-Alert Filtering, WiFi, GPS Based, Apple CarPlay and Android Auto Compatible",749.95
2933,"Uniden SDS100 True I/Q Digital Handheld Scanner, Designed for Improved Digital Performance in Weak-Signal and Simulcast Areas, Rugged / Weather Resistant JIS 4 Construction",699.99
1483,"Garmin Edge 1050®, Premium Cycling Computer, Vivid Color Touchscreen Display, Built-in Speaker, Advanced Training and Group Ride Features, Road Hazard Alerts",699.99


In [19]:
# Reestablecemos las opciones de visualización de pandas para ver títulos completos
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

# Definimos la lista de índices que identificamos como incorrectos
indices_falsos_perifericos = [
    634, 2456, 6417, 5163, 2314, 1641, 1796, 5145, 4582, 2380, # Pasivos/Cables
    4537, 5767, 3907, 5884, 4750, 4756, 2634, 2279, 2544, 7353, 318, 608, # Pasivos/Fundas/Calc
    4928, 3072, 1088, # Componentes PC
    1010, 2810, 2976, 1044, # Networking
    4196, 2649, 4960 # Audio/Video
]

# Ejemplo de visualización antes de mover
sample_peripherals.loc[indices_falsos_perifericos, 'original_title']


634                                                                                  UGREEN USB A to USB B Printer Cable 5ft - High-Speed for HP, Canon, Brother, Samsung, Dell, Epson, Lexmark, Xerox, and More
2456                      AINOPE Printer Cable USB A to USB B Priner Cable High-Speed Nylon Braided USB 2.0 MIDI Cable, Printer Cord to Computer for HP, Canon, Brother, Dell, Epson, Lexmark, Xerox, 6.6ft/Grey
6417                             AINOPE Printer Cable, [2-in-1] Printer USB B to USB C/A MIDI Cable High Speed Nylon Braided for MacBook Pro, HP Canon Brother Dell Epson Lexmark DAC,MIDI Keyboard, 6.6FT, Grey
5163                                                                              C2G 6FT Premium Replacement AC Power Cord - Durable Power Cable for TV, Computer, Monitor, Appliance & More (24240), Pack of 1
2314                                                                                            Amazon Basics PC Power Cord, 15 feet, AC Power Cord for Monitor, Com

In [73]:
df_final['category'].value_counts()

category
Accessories & Consumables           2791
Audio & Video Equipment             1200
Peripherals & Input Devices         1063
Networking & Smart Home              491
Displays & Mounting                  462
Cameras & Photography                457
PC Components (Core)                 455
Laptops & Chromebooks                334
Desktops, Workstations & Servers     188
Smartphones & Wearables              154
Tablets & E-Readers                   78
Gaming Systems & VR                   41
ERROR_API_FAIL                         3
Name: count, dtype: int64

In [1]:
import pandas as pd


ev3 = pd.read_csv('ev3_productos.csv')

In [2]:
ev3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7717 entries, 0 to 7716
Data columns (total 1 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   product_title  7717 non-null   object
dtypes: object(1)
memory usage: 60.4+ KB


In [25]:
def info_df(df):
    return pd.DataFrame({
    'Columna': df.columns,
    'No Nulos': df.notnull().sum().values,
    'Nulos': df.isnull().sum().values,
    'Tipo Python': df.dtypes.values,
    'Núm. valores': [len(df[col].unique()) for col in df.columns]
     })
info = info_df(df_final)
info

,Columna,No Nulos,Nulos,Tipo Python,Núm. valores
0,category,7717,0,object,13
1,market_tier,7714,3,object,5
2,condition,7714,3,object,3
3,is_premium_brand,7714,3,object,3
4,tech_generation,7714,3,object,5
5,ram_gb,626,7091,float64,19
6,storage_gb,853,6864,float64,64
7,size_value,1690,6027,float64,257
8,performance_value,1283,6434,float64,243
9,power_wattage,630,7087,float64,121


In [28]:
df_final['original_row_id']

0          0.0
1          1.0
2          2.0
3          3.0
4          4.0
         ...  
7712    7712.0
7713    7713.0
7714    7714.0
7715    7715.0
7716    7716.0
Name: original_row_id, Length: 7717, dtype: float64

In [30]:
df_final.drop(columns=['error_log', 'product_title', 'price_real', 'original_title', 'original_row_id', 'price_real'], inplace=True)

In [38]:
df_final.drop(columns=['product_category'], inplace=True)

In [31]:
info= info_df(df_final)
info

,Columna,No Nulos,Nulos,Tipo Python,Núm. valores
0,category,7717,0,object,13
1,market_tier,7714,3,object,5
2,condition,7714,3,object,3
3,is_premium_brand,7714,3,object,3
4,tech_generation,7714,3,object,5
5,ram_gb,626,7091,float64,19
6,storage_gb,853,6864,float64,64
7,size_value,1690,6027,float64,257
8,performance_value,1283,6434,float64,243
9,power_wattage,630,7087,float64,121


In [36]:
df_final[df_final['brand'] == 'Apple']

,category,market_tier,condition,is_premium_brand,tech_generation,ram_gb,storage_gb,size_value,performance_value,power_wattage,...,is_sponsored,buy_box_availability,sustainability_tags,has_coupon,discount_percentage,product_category,product_segment,log_original_price,log_purchased_last_month,log_total_reviews
6,Audio & Video Equipment,Premium,New,True,Current-Gen,NaN,NaN,NaN,NaN,NaN,...,Organic,0,0,0,0.00,Wearables,Baja,2.839078,9.210440,9.621655
8,Smartphones & Wearables,Premium,New,True,Current-Gen,NaN,NaN,40.0,NaN,NaN,...,Organic,0,0,0,0.00,Wearables,Media,4.991385,9.210440,9.460398
13,Audio & Video Equipment,Premium,New,True,Current-Gen,NaN,NaN,NaN,NaN,NaN,...,Organic,0,0,0,0.00,Wearables,Media,5.095222,9.210440,10.488019
14,Accessories & Consumables,Premium,New,True,Current-Gen,NaN,NaN,NaN,NaN,NaN,...,Organic,0,0,0,0.00,Small Gadget Accessories (Cases & Protectors),Media,4.300545,9.210440,10.274672
16,Audio & Video Equipment,Premium,New,True,Current-Gen,NaN,NaN,NaN,NaN,NaN,...,Organic,0,0,0,0.00,Wearables,Media,4.489872,9.210440,9.507998
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7166,Accessories & Consumables,Premium,New,True,Current-Gen,NaN,NaN,NaN,NaN,NaN,...,Organic,1,0,0,0.00,Tablets & E-readers (DEVICES ONLY),Media,4.110710,4.615121,7.610853
7440,Tablets & E-Readers,Premium,Renewed/Refurbished,True,Current-Gen,NaN,256.0,10.9,NaN,NaN,...,Organic,0,0,0,0.22,Tablets & E-readers (DEVICES ONLY),Media,6.142897,3.931826,4.836282
7623,Peripherals & Input Devices,Premium,Renewed/Refurbished,True,Current-Gen,NaN,NaN,NaN,NaN,NaN,...,Organic,0,0,0,0.09,Tablets & E-readers (DEVICES ONLY),Media,4.204693,5.303305,5.429346
7626,Tablets & E-Readers,Premium,Renewed/Refurbished,True,Current-Gen,NaN,64.0,7.9,NaN,NaN,...,Organic,0,0,0,0.00,Tablets & E-readers (DEVICES ONLY),Media,4.700480,5.303305,6.182085


In [39]:
df_final.to_csv('../../Datasets/evaluacion3.csv', index=False)